# 🦜🔗 LangChain Mastery for Software Engineers

Welcome. This notebook is designed for experienced software engineers transitioning into AI/LLM engineering. We bypass introductory Python concepts and focus on LangChain's architecture, abstractions, and production-ready patterns.

**Note:** LangChain has evolved significantly. This guide focuses on **LCEL (LangChain Expression Language)**, the modern, production-grade approach to building LLM applications, avoiding deprecated legacy chains.

### Prerequisites & Setup

Before running the code cells, install the required packages and set your API key.

In [ ]:
!pip install -qU langchain langchain-openai langchain-community langchain-core faiss-cpu python-dotenv tiktoken pydantic

import os
from google.colab import userdata

# Assuming you have added your OpenAI API key to Colab Secrets
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY') or "sk-your-key-here"

## 📑 Table of Contents

1. [**Level 1: Beginner - Core Fundamentals**](#level-1-beginner)
   * Concept 1: LCEL (LangChain Expression Language) & Chat Models
   * Concept 2: Prompt Templates & Output Parsers (Structured Data)
2. [**Level 2: Intermediate - RAG & State**](#level-2-intermediate)
   * Concept 3: Retrieval-Augmented Generation (RAG) Architecture
   * Concept 4: Memory & State Management
3. [**Level 5: Advanced - Agents & Production**](#level-3-advanced)
   * Concept 5: Tool Calling & Agents (ReAct)
   * Concept 6: Streaming, Async & Debugging (Callbacks)
4. [**Final Mini-Project**](#final-project)
   * Building an Autonomous Customer Support Triage System

<a id="level-1-beginner"></a>

## 🟢 Level 1: Beginner - Core Fundamentals

### Concept 1: LCEL & Chat Models

**Explanation:** LangChain's core abstraction is the `Runnable`. LCEL allows you to compose these runnables using the pipe operator `|`, similar to Unix pipelines. Under the hood, LCEL automatically handles synchronous, asynchronous (`ainvoke`), streaming (`stream`), and batch (`batch`) operations.

**Real World Use Case:** Standardizing interactions with different LLM providers (OpenAI, Anthropic, local models) behind a single microservice interface without rewriting the orchestration logic.

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 1. Initialize the Model
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# 2. Create a Prompt Template
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a senior DevOps engineer. Explain concepts concisely."),
    ("user", "Explain {topic} in 2 sentences.")
])

# 3. Build the LCEL Pipeline (Prompt -> LLM -> String Parser)
chain = prompt | llm | StrOutputParser()

# 4. Invoke the chain
response = chain.invoke({"topic": "Kubernetes Ingress"})
print(response)

**Expected Output:**

> Kubernetes Ingress is an API object that manages external access to the services in a cluster, typically providing HTTP and HTTPS routing. It acts as a reverse proxy, consolidating routing rules into a single resource to simplify traffic management and load balancing.

**Practice Exercise:** Modify the pipeline to include a second prompt that translates the output into Spanish. (Hint: `chain1 | translation_prompt | llm | StrOutputParser()`).

> 💡 **Interview Tip:** Be prepared to explain *why* LCEL is preferred over legacy chains. The answer: Native streaming, automatic parallelization (batching), and out-of-the-box tracing/observability via LangSmith.

### Concept 2: Prompt Templates & Output Parsers

**Explanation:**
LLMs output raw text natively. For software engineering, we need structured data (JSON/Objects). LangChain provides `OutputParsers` and `.with_structured_output()` to force LLMs to conform to Pydantic schemas, enabling type-safe downstream processing.

**Real World Use Case:**
Extracting entities (IP addresses, error codes, user IDs) from unstructured application logs to index in Elasticsearch.

In [ ]:
from pydantic import BaseModel, Field
from typing import List

# 1. Define the desired schema using Pydantic
class LogExtraction(BaseModel):
    error_code: str = Field(description="The HTTP or system error code")
    severity: str = Field(description="Severity level: INFO, WARN, ERROR, CRITICAL")
    affected_services: List[str] = Field(description="List of microservices mentioned")

# 2. Bind the schema to the model
structured_llm = llm.with_structured_output(LogExtraction)

prompt = ChatPromptTemplate.from_messages([
    ("system", "Extract diagnostic information from the following log snippet."),
    ("user", "{log_text}")
])

chain = prompt | structured_llm

log_text = "[2023-10-27 10:00:00] CRITICAL - 502 Bad Gateway in PaymentGatewayService. Auth-Service unreachable."
result = chain.invoke({"log_text": log_text})

print(type(result))
print(result.model_dump_json(indent=2))

**Expected Output:**

```
<class '__main__.LogExtraction'>
{
  "error_code": "502",
  "severity": "CRITICAL",
  "affected_services": [
    "PaymentGatewayService",
    "Auth-Service"
  ]
}
```

<a id="level-2-intermediate"></a>

## 🟡 Level 2: Intermediate - RAG & State

### Concept 3: Retrieval-Augmented Generation (RAG) Architecture

**Explanation:**
LLMs suffer from hallucinations and knowledge cutoffs. RAG solves this by retrieving relevant context from a Vector Database and injecting it into the prompt. The standard flow is: *Document Load -> Split -> Embed -> Store -> Retrieve -> Generate*.

**Real World Use Case:**
Building an internal Copilot that answers developer questions based on proprietary Confluence documentation and private GitHub repositories.

In [ ]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.runnables import RunnablePassthrough

# 1. Ingestion (Load & Split)
loader = WebBaseLoader("https://lilianweng.github.io/posts/2023-06-23-agent/")
docs = loader.load()
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
splits = text_splitter.split_documents(docs)

# 2. Embed & Store (Using FAISS for local, in-memory vector DB)
vectorstore = FAISS.from_documents(documents=splits, embedding=OpenAIEmbeddings())
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

# 3. Generation (RAG Chain)
template = """Answer the question based only on the following context:
{context}

Question: {question}"""
prompt = ChatPromptTemplate.from_template(template)

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# LCEL: Pass the question to the retriever, format docs, inject into prompt
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print(rag_chain.invoke("What is Task Decomposition?"))

**Expected Output:**

> Task decomposition is a process where a complex task is broken down into smaller, more manageable steps. This can be done by the LLM using simple prompting, using task-specific instructions, or with human inputs.

**Practice Exercise:** Change the `search_kwargs` in the retriever to use "mmr" (Maximal Marginal Relevance) instead of standard similarity search to improve context diversity.

> 💡 **Interview Tip:** A common RAG pitfall is the "Lost in the Middle" syndrome. Be ready to discuss advanced RAG techniques like Re-ranking (e.g., Cohere Re-rank), Parent Document Retrieval, and semantic chunking.

### Concept 4: Memory & State Management

**Explanation:**
LLMs are stateless APIs. To build conversational interfaces, we must manage the state (message history) externally. LangChain uses `RunnableWithMessageHistory` to wrap chains and automatically read/write to a session-specific message store.

**Real World Use Case:**
A persistent pair-programming chatbot embedded in an IDE that remembers the context of the user's codebase throughout a session.

In [ ]:
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.messages import HumanMessage

store = {}

def get_session_history(session_id: str) -> InMemoryChatMessageHistory:
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    ("placeholder", "{chat_history}"), # Automatically injected by history wrapper
    ("user", "{input}")
])

chain = prompt | llm | StrOutputParser()

# Wrap the chain with history management
with_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="chat_history",
)

# Call 1
print("Call 1:", with_history.invoke(
    {"input": "Hi, I'm analyzing a Python codebase."},
    config={"configurable": {"session_id": "user_123"}}
))

# Call 2
print("Call 2:", with_history.invoke(
    {"input": "What language did I say I was analyzing?"},
    config={"configurable": {"session_id": "user_123"}}
))

**Expected Output:**

> Call 1: Hello! I'd be happy to help you analyze your Python codebase. What specifically are you looking to do?
> Call 2: You mentioned that you are analyzing a Python codebase.

<a id="level-3-advanced"></a>

## 🔴 Level 3: Advanced - Agents & Production

### Concept 5: Tool Calling & Agents (ReAct)

**Explanation:**
Chains are deterministic DAGs (Directed Acyclic Graphs). **Agents** allow the LLM to act as the routing engine, deciding *which* tools to use, *when* to use them, and parsing the output to decide the next step (ReAct pattern: Reason + Act).

**Real World Use Case:**
An autonomous SRE agent that can query Datadog for CPU spikes, run an AWS CLI command to check EC2 instances, and automatically restart failing pods.

In [ ]:
from langchain_core.tools import tool
from langchain.agents import create_tool_calling_agent, AgentExecutor
from langchain import hub

# 1. Define custom tools using the @tool decorator
@tool
def check_server_status(server_id: str) -> str:
    """Useful for checking if a server is online or offline."""
    # In a real app, this would hit an API/DB
    mock_db = {"srv-01": "Online", "srv-02": "Offline", "srv-03": "High CPU"}
    return mock_db.get(server_id, "Server Not Found")

tools = [check_server_status]

# 2. Pull a standard tool-calling prompt from LangChain Hub
prompt = hub.pull("hwchase17/openai-functions-agent")

# 3. Create the agent and the executor loop
agent = create_tool_calling_agent(llm, tools, prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

# 4. Run the agent
result = agent_executor.invoke({"input": "Check the status of srv-02 and tell me what to do next."})
print("\nFinal Output:", result['output'])

**Expected Output:**

> *\[Agent trace will show it calling the tool `check_server_status` with `srv-02`\]*
> Final Output: The server 'srv-02' is currently offline. You should check the power supply, network connectivity, or try restarting the instance via your cloud console.

> 💡 **Interview Tip:** Understand the difference between `Agent` (the LLM logic that decides the next step) and `AgentExecutor` (the while-loop that actually executes the tool and feeds the result back to the Agent).

### Concept 6: Streaming, Async & Debugging

**Explanation:**
In production, TTFB (Time To First Token) is critical for UX. You must use asynchronous streaming. For debugging complex non-deterministic LLM pipelines, standard logging fails; LangChain's built-in callback handlers or LangSmith are required.

**Real World Use Case:**
Streaming responses to a React frontend via Server-Sent Events (SSE) while logging token usage metrics to Prometheus.

In [ ]:
import asyncio

async def run_streaming():
    prompt = ChatPromptTemplate.from_template("Write a 3 line poem about asynchronous programming in {language}.")
    chain = prompt | llm | StrOutputParser()
    
    # We use astream() for async streaming
    print("Streaming chunks:")
    async for chunk in chain.astream({"language": "Python"}):
        print(chunk, end="|", flush=True)

# Run the async function (Colab supports top-level await)
await run_streaming()

**Expected Output:**

> Streaming chunks:
> Async| awaits| the| call| to| run,|
> A| yield| of| time| until| it's| done,|
> In| event| loops| we| find| our| fun.|

<a id="final-project"></a>

## 🏆 Final Mini-Project: Autonomous Triage Agent

**Goal:** Build an agent that combines Memory, Tools, and structured reasoning to handle customer support tickets.

**Scenario:** The agent needs to answer questions. If the question requires looking up an order, it uses a tool. If it's a general policy question, it answers directly. It must remember the conversation.

In [ ]:
from langchain_core.messages import SystemMessage

# 1. Define the Tool
@tool
def get_order_status(order_id: str) -> str:
    """Fetches the shipping status of an order given its ID."""
    db = {"ORD-123": "Shipped - Arriving Tuesday", "ORD-999": "Processing"}
    return db.get(order_id, "Order not found. Check the ID.")

tools = [get_order_status]

# 2. Setup Agent architecture
system_prompt = SystemMessage(
    content="You are a helpful customer support bot. Use tools to check order statuses if asked."
)
prompt = ChatPromptTemplate.from_messages([
    system_prompt,
    ("placeholder", "{chat_history}"),
    ("user", "{input}"),
    ("placeholder", "{agent_scratchpad}"),
])

agent = create_tool_calling_agent(llm, tools, prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=False)

# 3. Add Memory
store = {}
def get_session(session_id: str):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

triage_agent = RunnableWithMessageHistory(
    agent_executor,
    get_session,
    input_messages_key="input",
    history_messages_key="chat_history",
)

# 4. Simulate a Conversation
session_config = {"configurable": {"session_id": "customer_a"}}

print("User: Hi, my name is Alex.")
res1 = triage_agent.invoke({"input": "Hi, my name is Alex."}, config=session_config)
print(f"Agent: {res1['output']}\n")

print("User: Can you check on order ORD-123?")
res2 = triage_agent.invoke({"input": "Can you check on order ORD-123?"}, config=session_config)
print(f"Agent: {res2['output']}\n")

print("User: What is my name again?")
res3 = triage_agent.invoke({"input": "What is my name again?"}, config=session_config)
print(f"Agent: {res3['output']}\n")

**Expected Output:**

> User: Hi, my name is Alex.
> Agent: Hello Alex! How can I assist you today?
>
> User: Can you check on order ORD-123?
> Agent: Your order ORD-123 has been shipped and is arriving on Tuesday. Let me know if you need anything else!
>
> User: What is my name again?
> Agent: Your name is Alex!